In [1]:
from lightkurve import DataCube, ErrorCube, DataFrame

%load_ext autoreload
%autoreload 2

In [2]:
from astropy.io import fits
import numpy as np
import pandas as pd

hdulist = fits.open(
    "./src/lightkurve/data/tess-s0001-4-2_84.291190_-80.469170_6x6_astrocut.fits"
)

# Prepare DataCube

In [3]:
flux_array = hdulist[1].data["FLUX"].astype(float)
flux_err_array = hdulist[1].data["FLUX_ERR"].astype(float)
time = hdulist[1].data["TIME"].astype(float)
time_corr = hdulist[1].data["TIMECORR"].astype(float)
c0, r0 = hdulist[1].header["1CRV4P"], hdulist[1].header["2CRV4P"]
row, col = np.arange(flux_array.shape[1]) + r0, np.arange(flux_array.shape[2]) + c0
aper = flux_array.mean(axis=0) > 10000
bkg_aper = flux_array.mean(axis=0) < 4000

time_mask = hdulist[1].data["QUALITY"] == 0

In [140]:
flux = DataCube(
    flux_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},
    col_indices={"pixel_column": col},
)

flux_err = ErrorCube(
    flux_err_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},
    col_indices={"pixel_column": col},
)

In [113]:
flux, flux_err

(📘 DataCube (50, 6, 6), 📕 ErrorCube (50, 6, 6))

In [122]:
flux

pixel_column,1628,1629,1630,1631,1632,1633
pixel_row,,,,,,
257,972.539368,965.366943,938.845215,1154.414795,1616.280029,2285.411133
258,2091.041260,2403.761475,42657.632812,65179.886719,8305.186523,7378.595703
259,4117.913574,6243.910156,100720.773438,101675.906250,42470.730469,8941.966797
260,5998.216797,29425.457031,102240.070312,103746.945312,61337.554688,7088.634766
261,4179.137207,26175.996094,103074.070312,105171.984375,40686.078125,4181.030273
262,3493.345703,11072.824219,102138.585938,104423.625000,10715.317383,3497.708008


In [83]:
flux_err

pixel_column,1628,1629,1630,1631,1632,1633
pixel_row,,,,,,
257,0.876119,0.868199,0.852017,0.945679,1.120531,1.321546
258,1.265783,1.356540,5.598882,6.952911,2.503161,2.350773
259,1.763391,2.166304,8.639107,8.677429,5.671174,2.576680
260,2.113300,4.675353,8.696113,8.800759,6.741338,2.305950
261,1.774688,4.403429,8.761114,8.907879,5.498235,1.769044
262,1.621295,2.873620,8.711569,8.847847,2.836913,1.626671


# Tests

## Downsample

In [ ]:
flux.downsample(5)

In [ ]:
flux_err.downsample(5)

## Imposing an aperture mask where `mean`(flux of pixel) > 10,000

In [ ]:
flux[:, aper]

## Sum of flux in aperture, per cadence

In [ ]:
flux[:, aper].sum(axis=1)

## Sum of flux error in aperture, per cadence. 

In [ ]:
flux_err[:, aper].sum(axis=1)

# Check types when transformed

In [ ]:
type(flux), type(flux_err)

In [ ]:
type(flux[:, aper]), type(flux_err[:, aper])

In [ ]:
type(flux[:, 0, :]), type(flux_err[:, 0, :])
type(flux[:, :, 0]), type(flux_err[:, :, 0])

In [ ]:
type(flux[:, 0, 0]), type(flux_err[:, 0, 0])

In [ ]:
type(flux[:, aper].sum(axis=0)), type(flux_err[:, aper].sum(axis=0))
type(flux[:, aper].sum(axis=1)), type(flux_err[:, aper].sum(axis=1))

# Plot the light curve

In [ ]:
import matplotlib.pyplot as plt

time = np.asarray(flux.btjd)
bkg = flux[:, bkg_aper].mean(axis=1)
bkg_err = flux_err[:, bkg_aper].mean(axis=1)

bkg -= bkg.median()

lc = flux[:, aper].sum(axis=1)
lc_err = flux_err[:, aper].sum(axis=1)

plt.figure()
plt.title("Raw")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")
# plt.ylim(.88e6, .94e6)

lc = flux[:, aper].sum(axis=1) - (bkg * aper.sum())
lc_err = flux_err[:, aper].sum(axis=1) + (bkg_err * aper.sum())

plt.figure()
plt.title("Background Subtracted")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")
# plt.ylim(0.88e6, 0.94e6)

## Info

repo: https://github.com/tessgi/secret/tree/christina-sketch1/src/lightkurve

# Milestones;

- ✅ Break the Cube, Frame, and Series classes into their own modules (cube.py, frame.py, series.py)
- ✅ Convert these demo functions into tests inside the package to fully test the functionality as users of lightkurve would need it. As you find any tests (breaking or passing) that cover functionality users need, add them to the tests.
    - ✅ To do this you will need to choose a reasonable piece of test data we can ship with the package. Crucially this must be small (ideally kb). You can trim down some tesscut data using lightkurve to add to the package
    - ✅ Confirm adjusted data product is appropriate and sufficient for tests
    - ✅ Trim to 50 frames
    - ✅ Put in tests/data
- ✅ Take a look at the pandas documentation and see if we can safely quiet the user warning that we have for adding our convenience functions (e.g. `flux.cadence`)
    - 🗒️ Add convenience functions to _metadata namespace
    - ❗️ Don't add existing attributes to this namespace, it messes up pandas at the core level somehow
- Currently there is an aggregate method for time. Create a similar aggregate method that will instead aggregate on columns so that;
    - ✅ DataCube.spatial_aggregate(...) -> returns a lower resolution datacube. e.g. sums pixels in row and column direction to create a lower resolution image
    - ✅ DataFrame.spatial_aggregate(...) -> returns a lower resolution dataframe
    - ✅ DataSeries has no spatial axis to aggregate
    - Create functions for ErrorCube and ErrorFrame
    - ✅ downsample methods more similar to time downsample
    - Think about binning to specific time/space points
- ✅ Tests for ErrorCubes
- Documentation
- ✅ Cube `__repr__`
- Consider bleed columns, can these be accounted for easily in the DataCube framework?

- We'll come back to; can we have in lightkurve3 a way to take cubes, frames, series and convert to fits imagehdu, tablehdu, and columnhdu.

- See what happens if I give too many rows and columns
- Change header -> "meta" or something like that
- meta could return a dataclass that is indexable, could hold relevant fits file information when loaded in
- Folding - add index. Return a new object (deepcopy) unless inplace=True
- Check downsample error
- Think about how lightkurve will interact with these objects
- Think about data quality handling
- 

## Smaller test data

In [ ]:
from lksearch import TESSSearch
from astropy.coordinates import SkyCoord

search_input = (84.291190, -80.469170)
search_input = SkyCoord(*search_input, unit="deg", frame="icrs")
search_result = TESSSearch(search_input, hlsp=False)

# You can filter two different ways to get the same result.
# search_result.filter_table(pipeline='TESScut')
downloads_table = search_result.tesscut.filter_table(sector=1).download(TESScut_size=6)
hdulist = fits.open(downloads_table["Local Path"][0])

In [ ]:
hdulist = fits.open(
    "/Users/dkgiles/.lksearch/cache/mastDownload/TESSCut/tess-s0001-4-2_84.291190_-80.469170_6x6_astrocut.fits"
)

In [ ]:
hdulist[1].data = hdulist[1].data[:50]

In [ ]:
hdulist.writeto(
    "./src/lightkurve/data/tess-s0001-4-2_84.291190_-80.469170_6x6_astrocut.fits",
)

## Spatial aggregation

In [ ]:
flux

In [ ]:
flux_lowres = flux.spatial_aggregate(3, 3)
flux_lowres

#### change to 
`flux_lowres = flux.spatial_aggregate(factor=int)`

In [ ]:
flux_lowres = flux.spatial_aggregate(4, 4)
flux_lowres

In [ ]:
def single_cadence_frame(datacube, cadence):
    return DataFrame(
        datacube.to_array()[cadence],
        index=datacube.row[:: datacube.nrow],
        columns=datacube.column[: datacube.ncol],
    )

In [ ]:
single_cadence_frame(flux_lowres, 0)

In [ ]:
flux.downsample?

In [ ]:
flux.downsample(5)

In [ ]:
df0 = single_cadence_frame(flux, 0)
df0

In [ ]:
print(f"Sum of first 2x2: {df0.iloc[:2, :2].sum().sum()}")
df0.iloc[:2, :2]

In [ ]:
df0.spatial_aggregate(3, 3)

### Spatial downsampling like the time downsampling

In [ ]:
flux.spatial_downsample(2)

In [ ]:
flux.sum().sum()

In [ ]:
flux.spatial_downsample(2).sum().sum()

In [ ]:
flux[:, :-1].spatial_downsample(2)

In [ ]:
flux[:, :-2].sum().sum()

In [ ]:
flux[:, :-1].spatial_downsample(2).sum().sum()

In [ ]:
flux[:, :, :-1].spatial_downsample(2)

In [ ]:
flux[:, :, :-2].sum().sum()

In [ ]:
flux[:, :, :-1].spatial_downsample(2).sum().sum()

In [ ]:
flux.spatial_downsample(2).to_array()[0]

In [ ]:
def single_cadence_frame(datacube, cadence):
    indices = [f"{i[0]} {i[1]}" for i in zip(datacube.index.names, datacube.index[0])]
    str_index = "; ".join(indices)
    return pd.DataFrame(
        datacube.to_array()[cadence],
        index=pd.Series(datacube.row[:: datacube.nrow], name="row"),
        columns=pd.Series(datacube.column[: datacube.ncol], name="column"),
    ).style.set_caption(str_index)

In [ ]:
f0 = single_cadence_frame(flux, 0)
f0

In [ ]:
f0.style.set_caption("name")

In [ ]:
f0.index.name = "row"

In [ ]:
f0.columns.name = "column"

In [ ]:
f0

In [ ]:
[f"{i[0]}, {i[1]}" for i in zip(flux.index.names, flux.index[0])]

In [ ]:
flux.to_dataframe(0, 0)

In [ ]:
flux.index[0]

In [ ]:
c = 0

In [ ]:
with np.printoptions(edgeitems=2, threshold=5):
    print(flux)
    for c in [0, -1]:
        indices = [
            (flux.index.names[i], flux.index[c][i])
            for i in range(len(flux.index.names))
        ]
        print(f"""{indices}\n{flux.to_array()[c]}\n...""")

In [ ]:
with pd.option_context("display.max_columns", 4):
    print(flux)
    for c in [0, -1]:
        indices = [
            (flux.index.names[i], flux.index[c][i])
            for i in range(len(flux.index.names))
        ]
        print(f"""{indices}\n{single_cadence_frame(flux, c)}\n...""")

In [ ]:
pd.set_option("display.max.columns", 4)

In [ ]:
print(repr(single_cadence_frame(flux, 0)))

In [ ]:
print(flux._repr_html_())

# Dummy data

In [38]:
ntime, nrow, ncol = 200, 10, 14
test_data = np.ones((ntime, nrow, ncol))
row, col = np.arange(test_data.shape[1]), np.arange(test_data.shape[2])
df = DataCube(test_data, row_indices={"row": row}, col_indices={"column": col})
df_err = ErrorCube(test_data, row_indices={"row": row}, col_indices={"column": col})

In [24]:
df, df_err

(📘 DataCube (200, 10, 14), 📕 ErrorCube (200, 10, 14))

In [36]:
df.info()

<class 'lightkurve.datacube.DataCube'>
MultiIndex: 200 entries, (0,) to (199,)
Columns: 140 entries, (0, 0, 0) to (139, 9, 13)
dtypes: float64(140)
memory usage: 237.0 KB


In [41]:
type(flux.info())

<class 'lightkurve.datacube.DataCube'>
MultiIndex: 50 entries, (0, 1325.324261183436, 1325.3235062838578) to (49, 1326.3450855298036, 1326.3443392075317)
Data columns (total 36 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (0, 257, 1628)   50 non-null     float64
 1   (1, 257, 1629)   50 non-null     float64
 2   (2, 257, 1630)   50 non-null     float64
 3   (3, 257, 1631)   50 non-null     float64
 4   (4, 257, 1632)   50 non-null     float64
 5   (5, 257, 1633)   50 non-null     float64
 6   (6, 258, 1628)   50 non-null     float64
 7   (7, 258, 1629)   50 non-null     float64
 8   (8, 258, 1630)   50 non-null     float64
 9   (9, 258, 1631)   50 non-null     float64
 10  (10, 258, 1632)  50 non-null     float64
 11  (11, 258, 1633)  50 non-null     float64
 12  (12, 259, 1628)  50 non-null     float64
 13  (13, 259, 1629)  50 non-null     float64
 14  (14, 259, 1630)  50 non-null     float64
 15  (15, 259, 1631)  50 non-nu

NoneType

In [37]:
df

column,0,1,2,3,4,5,6,7,8,9,...
row,,,,,,,,,,,
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
4,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
6,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
7,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...
8,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...


In [26]:
df_err

column,0,1,2,3,4,5,6,7,8,9,...
row,,,,,,,,,,,
0,1,1,1,1,1,1,1,1,1,1,...
1,1,1,1,1,1,1,1,1,1,1,...
2,1,1,1,1,1,1,1,1,1,1,...
3,1,1,1,1,1,1,1,1,1,1,...
4,1,1,1,1,1,1,1,1,1,1,...
5,1,1,1,1,1,1,1,1,1,1,...
6,1,1,1,1,1,1,1,1,1,1,...
7,1,1,1,1,1,1,1,1,1,1,...
8,1,1,1,1,1,1,1,1,1,1,...


In [ ]:
df.meta

In [ ]:
df_err.meta

In [ ]:
timedownsample = df.downsample(2)

In [30]:
df.downsample(3)

column,0,1,2,3,4,5,6,7,8,9,...
row,,,,,,,,,,,
0,3,3,3,3,3,3,3,3,3,3,...
1,3,3,3,3,3,3,3,3,3,3,...
2,3,3,3,3,3,3,3,3,3,3,...
3,3,3,3,3,3,3,3,3,3,3,...
4,3,3,3,3,3,3,3,3,3,3,...
5,3,3,3,3,3,3,3,3,3,3,...
6,3,3,3,3,3,3,3,3,3,3,...
7,3,3,3,3,3,3,3,3,3,3,...
8,3,3,3,3,3,3,3,3,3,3,...


In [15]:
(df.downsample(2).to_array() == 2).all()

True

In [153]:
(df.spatial_downsample(3) == 9).all().all()

True

In [39]:
df.spatial_downsample(3)

column,0,3,6,9
row,,,,
0,9.000000,9.000000,9.000000,9.000000
3,9.000000,9.000000,9.000000,9.000000
6,9.000000,9.000000,9.000000,9.000000


In [40]:
df_err.spatial_downsample(2)

column,0,2,4,6,8,10
row,,,,,,
0,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
2,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
4,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
6,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [ ]:
(df.downsample(3).to_array() == 3).all()

In [ ]:
(df.spatial_aggregate(5, 7).to_array().round() == 4).all()

In [22]:
df.spatial_aggregate(5, 7)
# needs offset to pretend row and column indices are spatial locations
# after aggregating once, offset -> 0
# If real spatial data is given (RA/dec), no offset
# RA and Dec Test

column,0.500000,2.500000,4.500000,6.500000,8.500000,10.500000,12.500000
row,,,,,,,
0.500000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
2.500000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
4.500000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
6.500000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
8.500000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000


In [ ]:
(df.spatial_downsample(2).to_array() == 4).all()

In [ ]:
(
    df[:, :, :-1].spatial_downsample(2).to_array()
    == df[:, :, :-2].spatial_downsample(2).to_array()
).all()

In [ ]:
(df_err.downsample(4).to_array() == 2).all()

In [ ]:
(df_err.downsample(9).to_array() == 3).all()

In [ ]:
(df_err.to_array()[0][0] ** 2).sum()

In [ ]:
df_err

In [ ]:
df_err.spatial_aggregate(5, 7)

In [ ]:
assert (df_err.spatial_aggregate(5, 7).to_array().round() == 2).all()

In [ ]:
flux[:, [1, 2, 3], [1, 2, 3]]

# Folding

In [157]:
flux.fold(0.3, inplace=True)
flux

pixel_column,1628,1629,1630,1631,1632,1633
pixel_row,,,,,,
257,972.539368,965.366943,938.845215,1154.414795,1616.280029,2285.411133
258,2091.041260,2403.761475,42657.632812,65179.886719,8305.186523,7378.595703
259,4117.913574,6243.910156,100720.773438,101675.906250,42470.730469,8941.966797
260,5998.216797,29425.457031,102240.070312,103746.945312,61337.554688,7088.634766
261,4179.137207,26175.996094,103074.070312,105171.984375,40686.078125,4181.030273
262,3493.345703,11072.824219,102138.585938,104423.625000,10715.317383,3497.708008
